In [ ]:
# datasets
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# control utils
from lerobot.utils.utils import log_say

# robot configs
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# utils
from dotenv import load_dotenv
from lerobot.utils.robot_utils import busy_wait

import time

# my code
from scripts.utils.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

c:\Users\jonathan\.conda\envs\lerobot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

This is used to replay the recorded dataset on the actual robot!

Connect to robot:

In [3]:
robot_config = SO101FollowerConfig(
    port="COM7",
    id="so_101_follower",
    calibration_dir = CALIBS_DIR
)

robot = SO101Follower(robot_config)
robot.connect()

ConnectionError: 
Could not connect on port 'COM7'. Make sure you are using the correct port.
Try running `python -m lerobot.find_port`


Load data:

In [ ]:
episode_idx = 0
REPO_NAME = 'so101_table_leg_assembly'
HF_NAME = 'jonathm126'
dataset = LeRobotDataset(f"{HF_NAME}/{REPO_NAME}", episodes=[episode_idx], root=f"{DATASETS_DIR}\\{REPO_NAME}")

RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6871206b-69f00c1a7aa82b5a67aaf13a;f6acfe57-f0e5-44a3-9609-6f53df83e214)

Repository Not Found for url: https://huggingface.co/api/datasets/jonathm126/so101_table_leg_assembly/refs.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Invalid username or password.

Dataset:

In [ ]:
actions = dataset.hf_dataset.select_columns("action")

In [ ]:
log_say(f"Replaying episode {episode_idx}")
for idx in range(dataset.num_frames):
    t0 = time.perf_counter()

    action = {
        name: float(actions[idx]["action"][i]) for i, name in enumerate(dataset.features["action"]["names"])
    }
    robot.send_action(action)

    busy_wait(1.0 / dataset.fps - (time.perf_counter() - t0))

robot.disconnect()